# LightRet â€” Stage 3: Noisy-Student NER Fine-Tuning

**Goal**: Fine-tune LightRet + BiLSTM NER head on CoNLL-2003 with character-level noise.  
- **Teacher**: LightRet from Stage 2, frozen, processes **clean** text  
- **Student**: same architecture + BiLSTM NER head, processes **noisy** text  
- **BiGRU** acts as a denoising filter â€” maps noisy RetVec embeddings back toward clean space  
- **Loss**: `Î²Â·L_class + (1âˆ’Î²)Â·L_distill`  where `Î² = 0.5`

Dynamic shift alignment (`build_word_alignment`) bridges spaces deleted/inserted by noise  
to keep teacher/student token positions correctly paired.

## Required Kaggle Datasets (add before running)
| Dataset name | Contents |
|---|---|
| `lightret-source` | Full project folder (src/, train_*.py, etc.) |
| `lightret-weights` | `retvec_v1_weights.npz` |
| `lightret-stage2-ckpt` | `lightret_stage2.pt` (output of Stage 2 notebook) |

> **Internet**: ON â€” downloads CoNLL-2003 from HuggingFace.

## Expected runtime (T4 GPU)
- CoNLL-2003 train: 14,041 sentences â€” very fast per epoch
- Per epoch: ~3â€“5 min
- Total 10 epochs: ~45 min

## Output
`/kaggle/working/weights/lightret_stage3.pt` â€” final LightRet backbone  
NER head weights are also saved separately as `ner_head_stage3.pt`.

In [ ]:
# â”€â”€ 1. Install packages â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
!pip install -q datasets seqeval

In [ ]:
# â”€â”€ 2. Setup working directory â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import os, shutil, pathlib

# !! Adjust dataset names if yours differ !!
SOURCE_DATASET   = '/kaggle/input/lightret-source'
WEIGHTS_DATASET  = '/kaggle/input/lightret-weights'
STAGE2_DATASET   = '/kaggle/input/lightret-stage2-ckpt'

WORK = pathlib.Path('/kaggle/working')

src_dst = WORK / 'src'
if src_dst.exists():
    shutil.rmtree(src_dst)
shutil.copytree(f'{SOURCE_DATASET}/src', str(src_dst))

shutil.copy(f'{WEIGHTS_DATASET}/retvec_v1_weights.npz', str(WORK / 'retvec_v1_weights.npz'))

(WORK / 'weights').mkdir(exist_ok=True)
shutil.copy(f'{STAGE2_DATASET}/lightret_stage2.pt', str(WORK / 'weights' / 'lightret_stage2.pt'))

os.chdir(str(WORK))
print('Weights dir:', sorted(os.listdir('weights')))

In [ ]:
# ── Patch stale config.py before importing ───────────────────────────────────
# Fixes uploaded dataset having old dataset name (conll2003 -> eriktks/conll2003)
p = '/kaggle/working/src/config.py'
with open(p) as f:
    txt = f.read()
txt = txt.replace('"conll2003"', '"eriktks/conll2003"')
with open(p, 'w') as f:
    f.write(txt)
print('config.py patched — conll dataset:', end=" ")
[print(l.strip()) for l in txt.splitlines() if "conll2003" in l and "DATASET" in l]

In [ ]:
# â”€â”€ 3. Verify GPU â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# â”€â”€ 4. Imports â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import sys
sys.path.insert(0, '/kaggle/working')

import src.config as cfg
from src.config import DEVICE, STAGE2_CKPT, STAGE3_CKPT, NER_LABELS, NER_ID2LABEL
from src.models.lightret import LightRet
from src.models.ner_head import NERHead
from src.data.dataset import NERDataset, ner_collate
from src.losses import stage3_loss

print('STAGE2_CKPT:', STAGE2_CKPT, '| exists:', STAGE2_CKPT.exists())
print('NER labels:', NER_LABELS)

In [ ]:
# â”€â”€ 5. Hyperparameters â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Uncomment to adjust
# cfg.STAGE3_EPOCHS     = 5
# cfg.STAGE3_BATCH_SIZE = 16
# cfg.STAGE3_BETA       = 0.5   # balance L_class vs L_distill

print(f'Epochs     : {cfg.STAGE3_EPOCHS}')
print(f'Batch size : {cfg.STAGE3_BATCH_SIZE}')
print(f'LR         : {cfg.STAGE3_LR}')
print(f'Beta       : {cfg.STAGE3_BETA}  (beta*L_class + (1-beta)*L_distill)')

In [ ]:
# â”€â”€ 6. Load CoNLL-2003 â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
train_ds = NERDataset(split='train',      apply_noise_aug=True)
valid_ds = NERDataset(split='validation', apply_noise_aug=False)
print(f'Train: {len(train_ds):,}  Val: {len(valid_ds):,}')

# Quick peek at a noisy example
sample = train_ds[0]
print('\nSample:')
print('  clean :', sample['clean_words'][:8])
print('  noisy :', sample['noisy_words'][:8])
print('  labels:', [NER_ID2LABEL[l] for l in sample['noisy_labels'][:8]])

In [ ]:
# â”€â”€ 7. Build models â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import torch.nn as nn

# Teacher: LightRet from Stage 2 (frozen, no projector)
teacher = LightRet.from_stage2_checkpoint(str(STAGE2_CKPT)).to(DEVICE)
teacher.eval()
for p in teacher.parameters():
    p.requires_grad_(False)
print('Teacher (frozen) â€” projector dropped')

# Student: same backbone (trainable) + NER head
student  = LightRet.from_stage2_checkpoint(str(STAGE2_CKPT)).to(DEVICE)
ner_head = NERHead().to(DEVICE)

trainable = (
    [p for p in student.parameters()  if p.requires_grad]
    + list(ner_head.parameters())
)
print(f'Student trainable params: {sum(p.numel() for p in trainable):,}')

In [ ]:
# â”€â”€ 8. Training â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import math
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR

def make_scheduler(opt, warmup, total):
    def lr_lambda(s):
        if s < warmup:
            return s / max(1, warmup)
        p = (s - warmup) / max(1, total - warmup)
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * p)))
    return LambdaLR(opt, lr_lambda)

train_loader = DataLoader(train_ds, batch_size=cfg.STAGE3_BATCH_SIZE,
                          shuffle=True, collate_fn=ner_collate, num_workers=2, pin_memory=True)
valid_loader = DataLoader(valid_ds, batch_size=cfg.STAGE3_BATCH_SIZE,
                          shuffle=False, collate_fn=ner_collate, num_workers=2)

optimizer   = AdamW(trainable, lr=cfg.STAGE3_LR, weight_decay=0.01)
total_steps = cfg.STAGE3_EPOCHS * len(train_loader)
scheduler   = make_scheduler(optimizer, cfg.STAGE3_WARMUP_STEPS, total_steps)

train_history = []   # (total, lc, ld) per epoch
val_history   = []
best_val      = float('inf')

for epoch in range(1, cfg.STAGE3_EPOCHS + 1):
    student.train(); ner_head.train()
    et = elc = eld = 0.0

    for step, batch in enumerate(train_loader, 1):
        clean_words   = batch['clean_words']
        noisy_words   = batch['noisy_words']
        noisy_labels  = batch['noisy_labels'].to(DEVICE)
        alignment     = batch['alignment']
        noisy_lengths = batch['noisy_lengths']

        with torch.no_grad():
            h_teacher = teacher(clean_words)

        h_student = student(noisy_words)
        logits    = ner_head(h_student, noisy_lengths)
        total, lc, ld = stage3_loss(
            logits, noisy_labels, h_teacher, h_student, alignment, cfg.STAGE3_BETA
        )

        optimizer.zero_grad()
        total.backward()
        nn.utils.clip_grad_norm_(trainable, 1.0)
        optimizer.step()
        scheduler.step()

        et += total.item(); elc += lc.item(); eld += ld.item()

    n = len(train_loader)
    train_history.append((et/n, elc/n, eld/n))

    # Validation: cross-entropy on clean labels
    student.eval(); ner_head.eval()
    vl = 0.0
    with torch.no_grad():
        for batch in valid_loader:
            nw  = batch['noisy_words']
            nl  = batch['noisy_labels'].to(DEVICE)
            nln = batch['noisy_lengths']
            h   = student(nw)
            lg  = ner_head(h, nln)
            B, L, C = lg.shape
            vl += F.cross_entropy(lg.reshape(-1,C), nl.reshape(-1),
                                  ignore_index=cfg.NER_IGNORE_INDEX).item()
    val_loss = vl / len(valid_loader)
    val_history.append(val_loss)

    t, lc_, ld_ = train_history[-1]
    print(f'E{epoch:02d}  train_total={t:.4f}  lc={lc_:.4f}  ld={ld_:.4f}  val_ce={val_loss:.4f}')

    if val_loss < best_val:
        best_val = val_loss
        torch.save(student.state_dict(),  str(STAGE3_CKPT))
        torch.save(ner_head.state_dict(), str(cfg.WEIGHTS_DIR / 'ner_head_stage3.pt'))
        print(f'  -> saved checkpoint')

print(f'\nDone. Best val CE: {best_val:.4f}')

In [ ]:
# â”€â”€ 9. seqeval F1 evaluation â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
from seqeval.metrics import classification_report
from src.config import NER_ID2LABEL, NER_IGNORE_INDEX

# Reload best checkpoint
student.load_state_dict(torch.load(str(STAGE3_CKPT), map_location=DEVICE, weights_only=True))
ner_head.load_state_dict(torch.load(str(cfg.WEIGHTS_DIR / 'ner_head_stage3.pt'),
                                    map_location=DEVICE, weights_only=True))
student.eval(); ner_head.eval()

all_preds, all_true = [], []
with torch.no_grad():
    for batch in valid_loader:
        nw  = batch['noisy_words']
        nl  = batch['noisy_labels'].to(DEVICE)
        nln = batch['noisy_lengths']
        h   = student(nw)
        lg  = ner_head(h, nln)               # (B, L, C)
        preds = lg.argmax(-1).cpu()          # (B, L)

        for i, length in enumerate(nln):
            p = [NER_ID2LABEL[preds[i, j].item()] for j in range(length)]
            t = [NER_ID2LABEL[nl[i, j].item()]    for j in range(length)
                 if nl[i, j].item() != NER_IGNORE_INDEX]
            if len(t) == length:   # skip if any labels were masked
                all_preds.append(p)
                all_true.append(t)

print(classification_report(all_true, all_preds))

In [ ]:
# â”€â”€ 10. Loss curves â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import matplotlib.pyplot as plt

epochs = range(1, len(train_history)+1)
total_ = [x[0] for x in train_history]
lc_    = [x[1] for x in train_history]
ld_    = [x[2] for x in train_history]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs, total_, label='total', marker='o')
axes[0].plot(epochs, lc_,    label='L_class', linestyle='--')
axes[0].plot(epochs, ld_,    label='L_distill', linestyle=':')
axes[0].set_title('Stage 3 train loss'); axes[0].legend()

axes[1].plot(epochs, val_history, color='orange', marker='s')
axes[1].set_title('Val cross-entropy')

plt.tight_layout()
plt.savefig('/kaggle/working/stage3_loss.png', dpi=100)
plt.show()

import os
for f in ['lightret_stage3.pt', 'ner_head_stage3.pt']:
    p = cfg.WEIGHTS_DIR / f
    if p.exists():
        print(f'{f}: {os.path.getsize(p)/1e6:.1f} MB')

In [ ]:
# â”€â”€ 11. Quick inference demo â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import torch

def predict(sentence: str) -> list[tuple[str, str]]:
    """Run NER on a raw sentence string. Returns [(word, label), ...]"""
    words = sentence.split()
    with torch.no_grad():
        h = student([words]).to(DEVICE)
        lg = ner_head(h, [len(words)])
    preds = lg.argmax(-1)[0].cpu().tolist()
    return [(w, NER_ID2LABEL[p]) for w, p in zip(words, preds[:len(words)])]

examples = [
    'London is the capital of the United Kingdom',
    'Micros0ft was founded by Bill Gates in Seattle',   # with OCR noise
    'Barack Obama visited Paris last week',
]
for sent in examples:
    result = predict(sent)
    print(f'\n  Input: {sent}')
    for word, label in result:
        tag = f' [{label}]' if label != 'O' else ''
        print(f'    {word}{tag}')